# 第 7 章 分類モデルの評価指標

正解率が役に立たない場面を出発点に、混同行列・適合率・再現率・F1・AUC を確かめます。

対応する記事: [第 7 章 分類モデルの評価指標（Kotlin Notebook の言語版）](../../../docs/article/grokking-machine-learning/kotlin/ch07.md)

実装本体: `apps/grokking-ml-kotlin/src/`

## セットアップ

実装本体をビルドした JAR を読み込みます。**ノートブックにコードを複製せず、記事と同じ実装をそのまま使います。**

先に JAR を作っておいてください。

```bash
cd apps/grokking-ml-kotlin
./gradlew jar
```

IntelliJ IDEA の Kotlin Notebook プラグイン、または [Kotlin Jupyter カーネル](https://github.com/Kotlin/kotlin-jupyter) で開きます。

```bash
pip install kotlin-jupyter-kernel
jupyter lab notebooks/
```

In [1]:
@file:DependsOn("../build/libs/grokking-ml-kotlin-0.1.0.jar")

import ch07.*

## 正解率 99% の役立たずモデル

1000 人に 10 人が罹る病気を、**全員「陰性」と判定する** モデルです。正解率は 99% ですが、病人を 1 人も見つけられません。

**正解率は、陽性と陰性の数が偏っているときに壊れます。**

In [2]:
val sickLabels = List(10) { 1 } + List(990) { 0 }
val alwaysHealthy = List(1000) { 0 }

val matrix = confusionMatrix(sickLabels, alwaysHealthy)
println(matrix)
println("正解率 %.3f".format(accuracy(matrix)))
println("再現率 %.3f  ← 病人を 1 人も見つけられていない".format(recall(matrix)))

ConfusionMatrix(truePositives=0, falsePositives=0, falseNegatives=10, trueNegatives=990)


正解率 0.990
再現率 0.000  ← 病人を 1 人も見つけられていない


## 適合率と再現率はトレードオフ

逆に **全員を「陽性」と判定** すると、見逃しはゼロ（再現率 1.0）ですが、陽性と言った 1000 件のうち当たりは 10 件だけです。**片方の指標だけを追うと、必ずもう一方が壊れます。**

In [3]:
val aggressive = confusionMatrix(sickLabels, List(1000) { 1 })

println("%-12s %8s %8s %8s %8s".format("モデル", "正解率", "適合率", "再現率", "F1"))
listOf("全員陰性" to matrix, "全員陽性" to aggressive).forEach { (name, m) ->
    println("%-12s %8.3f %8.3f %8.3f %8.3f".format(name, accuracy(m), precision(m), recall(m), f1Score(m)))
}

モデル               正解率      適合率      再現率       F1


全員陰性            0.990    0.000    0.000    0.000


全員陽性            0.010    0.010    1.000    0.020


## F1 は調和平均

適合率 1.0・再現率 0.1 のモデルは、算術平均なら 0.55 と「まあまあ」に見えます。**F1 は 0.18 です。片方が壊れているモデルを、平均で誤魔化させません。**

In [4]:
val unbalanced = ConfusionMatrix(truePositives = 1, falsePositives = 0, falseNegatives = 9, trueNegatives = 90)

println("適合率 %.3f  再現率 %.3f".format(precision(unbalanced), recall(unbalanced)))
println("算術平均 %.3f".format((precision(unbalanced) + recall(unbalanced)) / 2))
println("F1       %.3f  ← 偏りを強く罰する".format(f1Score(unbalanced)))

適合率 1.000  再現率 0.100
算術平均 0.550
F1       0.182  ← 偏りを強く罰する


## F ベータで重視する側を選ぶ

病気の見逃しを避けたいなら `beta > 1`（再現率重視）、迷惑メール判定で誤検知を避けたいなら `beta < 1`（適合率重視）です。**指標そのものを目的に合わせて調整できます。**

In [5]:
val sample = ConfusionMatrix(truePositives = 3, falsePositives = 1, falseNegatives = 2, trueNegatives = 4)
println("適合率 %.3f  再現率 %.3f".format(precision(sample), recall(sample)))

listOf(0.5, 1.0, 2.0).forEach { beta ->
    println("F%s = %.4f".format(beta, fBetaScore(sample, beta = beta)))
}

適合率 0.750  再現率 0.600
F0.5 = 0.7143
F1.0 = 0.6667
F2.0 = 0.6250


## AUC は閾値に依存しない

AUC は **「陽性を陰性より高くランク付けできた組の割合」** です。確率が両極に分かれていても中央に固まっていても、**順位が同じなら AUC は同じ** になります。

In [6]:
val cases = listOf(
    Triple("完全な順位", listOf(1, 1, 0, 0), listOf(0.9, 0.8, 0.2, 0.1)),
    Triple("3/4 が正しい順", listOf(1, 0, 1, 0), listOf(0.8, 0.6, 0.4, 0.2)),
    Triple("情報なし", listOf(1, 0, 0, 1), listOf(0.8, 0.6, 0.4, 0.2)),
    Triple("完全に逆", listOf(0, 0, 1, 1), listOf(0.9, 0.8, 0.2, 0.1)),
)
cases.forEach { (name, ls, ps) -> println("%-16s AUC = %.3f".format(name, auc(ls, ps))) }

println()
println("確率を圧縮しても AUC は変わらない:")
println(" 両極  " + auc(listOf(1, 1, 0, 0), listOf(0.99, 0.98, 0.02, 0.01)))
println(" 中央  " + auc(listOf(1, 1, 0, 0), listOf(0.55, 0.54, 0.46, 0.45)))

完全な順位            AUC = 1.000
3/4 が正しい順        AUC = 0.750


情報なし             AUC = 0.500
完全に逆             AUC = 0.000



確率を圧縮しても AUC は変わらない:
 両極  1.0
 中央  1.0


## 試してみる: 閾値を動かす

**閾値はモデルの性能ではなく、運用上の選択です。** 下げれば再現率が上がり、適合率が下がります。

In [7]:
val labels = listOf(1, 1, 0, 0)
val probabilities = listOf(0.9, 0.4, 0.6, 0.1)

println("%6s %-16s %8s %8s".format("閾値", "予測", "適合率", "再現率"))
listOf(0.2, 0.5, 0.8).forEach { threshold ->
    val predictions = predictionsAtThreshold(probabilities, threshold)
    val m = confusionMatrix(labels, predictions)
    println("%6s %-16s %8.3f %8.3f".format(threshold, predictions, precision(m), recall(m)))
}

    閾値 予測                    適合率      再現率


   0.2 [1, 1, 1, 0]        0.667    1.000
   0.5 [1, 0, 1, 0]        0.500    0.500


   0.8 [1, 0, 0, 0]        1.000    0.500
